# 06 · Comparação Pantanal × Cerrado e narrativa

Reúne os resultados dos dois biomas lado a lado e fecha a história do trabalho.

**Pré-requisito:** notebook 03 rodado (gera `focos_com_clima.parquet` e `painel_estacao_dia.parquet` no Drive). Este notebook lê esses arquivos e não depende dos notebooks 04 e 05.

Todas as figuras são salvas em `MyDrive/figuras/` (arquivos `06_*.png`) para usar nos slides.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from google.colab import drive

drive.mount('/content/drive')

BIOMAS = ['Pantanal', 'Cerrado']

RAIO_KM = 100   # raio principal da junção (README)
BASE     = '/content/drive/MyDrive'
F_FOCOS  = f'{BASE}/datasets_queimadas/focos_com_clima.parquet'      # saída do notebook 03
F_PAINEL = f'{BASE}/datasets_queimadas/painel_estacao_dia.parquet'   # saída do notebook 03
SHP_ZIP  = f'{BASE}/datasets_inmet/dmRgNtTrBmBr_v2025_vetor_e250K.zip'  # ajuste se o zip do IBGE estiver em outra pasta
FIG_DIR  = f'{BASE}/figuras'
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
COR = {'Pantanal': '#2a9d8f', 'Cerrado': '#e76f51'}
MESES = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']


def salvar(fig, nome):
    fig.savefig(f'{FIG_DIR}/{nome}.png', dpi=150, bbox_inches='tight')


# Área dos biomas: calculada dos polígonos do IBGE (projeção de área equivalente).
# Se o shapefile não puder ser lido, usa valores de referência do IBGE (km²).
AREA_REF = {'Pantanal': 150355.0, 'Cerrado': 2036448.0}

def areas_biomas():
    try:
        import geopandas as gpd
        g = gpd.read_file(f'zip://{SHP_ZIP}!dmRgNtTrBmBr_v2025_dominios_vetor_e250K.shp')
        g = g.to_crs('ESRI:102033')                       # Albers América do Sul (área equivalente)
        g['km2'] = g.geometry.area / 1e6
        a = g.groupby('nm_bm')['km2'].sum()
        out = {b: float(a[a.index.str.lower() == b.lower()].iloc[0]) for b in AREA_REF}
        print('Área calculada dos polígonos do IBGE (km²):', {k: round(v) for k, v in out.items()})
        return out, g
    except Exception as e:
        print(f'Não consegui ler o shapefile ({type(e).__name__}); usando áreas de referência do IBGE:', AREA_REF)
        return AREA_REF, None

AREA, GBIOMAS = areas_biomas()


## Carga dos dados

In [ ]:
focos_all  = pd.read_parquet(F_FOCOS)
painel_all = pd.read_parquet(F_PAINEL)
for d in (focos_all, painel_all):
    d['Data'] = pd.to_datetime(d['Data'])
focos_all['Ano'] = focos_all['Data'].dt.year
focos_all['Mes'] = focos_all['Data'].dt.month

# Painel por bioma: estações a até 100 km do bioma; contagem = focos DESTE bioma em volta da estação
def painel_bioma(b):
    q = painel_all[painel_all[f'dist_{b.lower()}_km'] <= RAIO_KM].copy()
    q['Bioma'] = b
    q['n_focos'] = q[f'n_focos_r100_{b}']
    q['n_focos_50'] = q[f'n_focos_r50_{b}']
    q['Teve_foco'] = q['n_focos'] > 0
    return q

pb = pd.concat([painel_bioma(b) for b in BIOMAS], ignore_index=True)
print(focos_all.groupby('Bioma').size().rename('focos').to_frame().join(pb.groupby('Bioma')['Codigo'].nunique().rename('estações ≤100 km')))


## 1. Volume

In [ ]:
# --- 1. Volume: contagem bruta e normalizada por área ---
vol = focos_all.groupby(['Ano', 'Bioma']).size().unstack()[BIOMAS]
vol_norm = vol.div(pd.Series(AREA)[BIOMAS]) * 1000

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, dado, tit, yl in [(ax[0], vol, 'Focos por ano (contagem bruta)', 'Focos'),
                         (ax[1], vol_norm, 'Focos por ano por 1.000 km² (normalizado pela área)', 'Focos / 1.000 km²')]:
    dado.plot.bar(ax=a, color=[COR[b] for b in dado.columns], width=0.75)
    a.set_title(tit); a.set_ylabel(yl); a.set_xlabel(''); a.tick_params(axis='x', rotation=0)
plt.tight_layout()
salvar(fig, '06_01_volume_bruto_vs_normalizado')
plt.show()

display(vol_norm.round(1))
print('Densidade média do período (focos/1.000 km²/ano):', (vol_norm.mean()).round(1).to_dict())
print('Razão Pantanal/Cerrado: bruto =', round(vol['Pantanal'].sum() / vol['Cerrado'].sum(), 2),
      '| normalizado =', round(vol_norm['Pantanal'].mean() / vol_norm['Cerrado'].mean(), 2))


## 2. Sazonalidade

In [ ]:
# --- 2. Sazonalidade: % dos focos do bioma em cada mês ---
mes = focos_all.groupby(['Bioma', 'Mes']).size().unstack(fill_value=0).reindex(columns=range(1, 13), fill_value=0)
pct = mes.div(mes.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 4.3))
for b in BIOMAS:
    ax.plot(MESES, pct.loc[b].values, marker='o', linewidth=2.5, color=COR[b], label=b)
ax.set_title('Em que meses cada bioma queima? (% dos focos do período)')
ax.set_ylabel('% dos focos do bioma'); ax.legend()
salvar(fig, '06_02_sazonalidade_comparada')
plt.show()

for b in BIOMAS:
    top = pct.loc[b].sort_values(ascending=False).head(3)
    print(f'{b}: pico em', ', '.join(f'{MESES[m - 1]} ({v:.1f}%)' for m, v in top.items()),
          f'| Ago–Out concentram {pct.loc[b, [8, 9, 10]].sum():.0f}% dos focos')


## 3. Anos atípicos

In [ ]:
# --- 3. Variabilidade entre anos: cada bioma contra a sua própria média ---
indice = vol / vol.mean()
cv = (vol.std() / vol.mean()).round(2)

fig, ax = plt.subplots(figsize=(8, 4.2))
for b in BIOMAS:
    ax.plot(indice.index.astype(str), indice[b].values, marker='o', linewidth=2.5, color=COR[b], label=b)
ax.axhline(1, color='gray', linestyle='--', linewidth=1)
ax.set_title('Focos por ano em relação à média do próprio bioma (1 = média)')
ax.set_ylabel('Índice'); ax.legend()
salvar(fig, '06_03_variabilidade_entre_anos')
plt.show()

print('Coeficiente de variação entre anos (desvio/média):', cv.to_dict())
for b in BIOMAS:
    print(f'{b}: ano de pico {vol[b].idxmax()} = {vol[b].max() / vol[b].drop(vol[b].idxmax()).median():.1f}x a mediana dos outros anos')


## 4. Intensidade (FRP)

In [ ]:
# --- 4. Intensidade: FRP por bioma ---
fr = focos_all.dropna(subset=['FRP'])
fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.boxplot(data=fr, x='Bioma', y='FRP', order=BIOMAS, hue='Bioma', palette=COR, fliersize=1, legend=False, ax=ax)
ax.set_yscale('log'); ax.set_ylabel('FRP (MW, escala log)'); ax.set_xlabel('')
ax.set_title('Intensidade do fogo (FRP) por bioma')
salvar(fig, '06_04_frp_por_bioma')
plt.show()

display(fr.groupby('Bioma')['FRP'].describe(percentiles=[.5, .9, .99]).round(1)[['count', 'mean', '50%', '90%', '99%', 'max']])


## 5. Clima × fogo

In [ ]:
# --- 5. Clima × fogo: os dois biomas lado a lado ---
pb['Faixa_seca'] = pd.cut(pb['Dias_sem_chuva'], [-1, 0, 6, 14, 29, 59, 10_000], labels=['0', '1–6', '7–14', '15–29', '30–59', '60+'])
pb['Faixa_umid'] = pd.cut(pb['Umid_min'], [-1, 20, 30, 40, 60, 101], labels=['<20', '20–30', '30–40', '40–60', '>60'])

fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
for a, col, tit in [(ax[0], 'Faixa_seca', 'dias sem chuva'), (ax[1], 'Faixa_umid', 'umidade mínima do dia (%)')]:
    t = pb.groupby(['Bioma', col], observed=True)['Teve_foco'].mean().mul(100).reset_index()
    sns.barplot(data=t, x=col, y='Teve_foco', hue='Bioma', hue_order=BIOMAS, palette=COR, ax=a)
    a.set_title(f'% de dias com foco por faixa de {tit}'); a.set_ylabel('% de estação-dias com foco'); a.set_xlabel('')
plt.tight_layout()
salvar(fig, '06_05_prob_foco_por_clima')
plt.show()

VARS = {'Dias_sem_chuva': 'Dias sem chuva', 'Umid_min': 'Umidade mínima (%)', 'Temp_max': 'Temperatura máxima (°C)', 'Precip_total': 'Chuva do dia (mm)'}
med = pb.groupby(['Bioma', 'Teve_foco'])[list(VARS)].median().round(1)
med.index = med.index.set_levels(['sem foco', 'com foco'], level=1)
print('Medianas nos dias sem foco vs com foco:'); display(med)

linhas = []
for b in BIOMAS:
    q = pb[pb['Bioma'] == b]
    for v, nome in VARS.items():
        for col, raio in [('n_focos', '100 km'), ('n_focos_50', '50 km')]:
            d = q[[v, col]].dropna()
            linhas.append({'Bioma': b, 'variável': nome, 'raio': raio, 'rho': round(spearmanr(d[v], d[col])[0], 3)})
rho = pd.DataFrame(linhas).pivot(index='variável', columns=['Bioma', 'raio'], values='rho')
print('Correlação de Spearman (nº de focos em volta da estação × variável):'); display(rho)


## 6. Cobertura das estações

In [ ]:
# --- 6. Cobertura de estações: ressalva importante da comparação ---
cob = focos_all.groupby('Bioma').agg(pct_focos_com_estacao_100km=('Tem_clima', 'mean'),
                                     pct_focos_com_estacao_50km=('Dentro_raio_teste', 'mean'),
                                     dist_mediana_km=('Dist_estacao_km', 'median')).loc[BIOMAS]
cob[['pct_focos_com_estacao_100km', 'pct_focos_com_estacao_50km']] *= 100
display(cob.round(1))

fig, ax = plt.subplots(figsize=(6.5, 4))
cob['pct_focos_com_estacao_100km'].plot.bar(ax=ax, color=[COR[b] for b in cob.index])
ax.set_ylabel('% dos focos'); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=0)
ax.set_title('Focos com estação INMET a até 100 km (no dia)')
salvar(fig, '06_06_cobertura_estacoes')
plt.show()


## 7. Mapa

In [ ]:
# --- 7. Mapa dos dois biomas ---
fig, ax = plt.subplots(figsize=(8, 8))
if GBIOMAS is not None:
    gb = GBIOMAS.to_crs('EPSG:4326')
    gb[gb['nm_bm'].str.lower().isin([b.lower() for b in BIOMAS])].boundary.plot(ax=ax, color='k', linewidth=0.8)
for b in BIOMAS:
    s = focos_all[focos_all['Bioma'] == b]
    s = s.sample(min(len(s), 30_000), random_state=0)
    ax.scatter(s['Longitude'], s['Latitude'], s=1, alpha=0.25, color=COR[b], label=f'focos {b} (amostra)')
est_all = painel_all.drop_duplicates('Codigo')
ax.scatter(est_all['Longitude'], est_all['Latitude'], s=10, marker='^', color='navy', label='estações INMET')
ax.set_title('Focos de calor (2020–2023) e estações INMET'); ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend(loc='lower left', markerscale=6); ax.set_aspect('equal')
salvar(fig, '06_07_mapa_biomas')
plt.show()


## 8. Resumo numérico

In [ ]:
# --- Tabela-resumo para a apresentação ---
resumo = pd.DataFrame({
    'Focos 2020–2023': vol.sum(),
    'Focos/1.000 km²/ano (média)': vol_norm.mean().round(1),
    'Ano de pico': vol.idxmax(),
    'FRP mediano (MW)': fr.groupby('Bioma')['FRP'].median().round(1),
    'Dias sem chuva (mediana) c/ foco': pb[pb['Teve_foco']].groupby('Bioma')['Dias_sem_chuva'].median(),
    'Dias sem chuva (mediana) s/ foco': pb[~pb['Teve_foco']].groupby('Bioma')['Dias_sem_chuva'].median(),
    '% focos com estação ≤100 km': cob['pct_focos_com_estacao_100km'].round(1),
}).loc[BIOMAS]
display(resumo.T)
resumo.T.to_csv(f'{FIG_DIR}/06_resumo_comparacao.csv')


## Narrativa: dois biomas, dois regimes de fogo

> **Antes de usar este texto:** os números abaixo vieram de uma reprodução dos dados feita fora do Colab. Confira cada número com as saídas das células acima do seu notebook e ajuste o que divergir.

**Volume.** O Cerrado concentra a imensa maioria dos focos em contagem bruta (234 mil contra 38,6 mil no Pantanal entre 2020 e 2023), mas ele é cerca de 13 vezes maior. Normalizando pela área, o Pantanal tem densidade média semelhante à do Cerrado (cerca de 64 contra 29 focos por 1.000 km² ao ano), ou seja, proporcionalmente queima mais.

**Sazonalidade.** Os dois biomas queimam na estação seca, com pico em setembro e concentração de cerca de dois terços dos focos entre agosto e outubro. No agregado os picos coincidem. O Cerrado tem pico em setembro todos os anos; no Pantanal o pico é em setembro em 2020 e 2021, mas foi em julho em 2022 e em novembro em 2023. Ou seja, a época geral é a mesma, mas o Pantanal varia mais de ano para ano, tanto em intensidade quanto em momento.

**Anos atípicos.** O Cerrado é estável entre os anos (queda gradual de 63,8 mil para 50,7 mil focos). O Pantanal é muito irregular: 2020 sozinho responde por 57% dos focos do período (22.116, cerca de 3,4 vezes a mediana dos outros anos), enquanto 2022 teve apenas 1.637. O regime de fogo do Pantanal é de anos extremos, o do Cerrado é de volume alto e constante.

**Clima.** Nos dois biomas, os dias com foco em volta da estação são mais secos, mais quentes e com umidade mínima menor (no Pantanal, mediana de 7 dias sem chuva e 28% de umidade mínima nos dias com foco, contra 1 dia e 43% nos dias sem foco; no Cerrado, 10 dias e 30% contra 2 dias e 44%). As correlações de Spearman têm o sinal esperado, mas são moderadas (em torno de 0,1 a 0,3 em módulo), e o mesmo resultado aparece com raio de 50 km. A chance de haver foco sobe com os dias sem chuva e com a queda da umidade nos dois biomas, com o mesmo gradiente. O nível mais baixo no Pantanal deve ser lido com cuidado, por causa da cobertura de estações (ver limitações). Clima ajuda a explicar quando o fogo é mais provável, mas está longe de explicar tudo.

**Intensidade.** O FRP mediano é maior no Pantanal que no Cerrado (cerca de 48 contra 31 MW), mostrando focos mais intensos por foco no Pantanal.

**Limitações que precisam aparecer na apresentação.**
- Cobertura de estações: no Pantanal, só cerca de 31% dos focos têm estação INMET a até 100 km no dia (no Cerrado, cerca de 79%), e a distância mediana até a estação é de 130 km (contra 65 km). O resultado de clima do Pantanal vale para a parte do bioma próxima às estações.
- Associação não é causalidade: muitos incêndios têm origem humana (manejo agrícola, desmatamento).
- Focos de calor não são área queimada, e usamos um único satélite (AQUA_M-T) para a série ser comparável.
- `RiscoFogo`, `DiaSemChuva` e `Precipitacao` do INPE são estimativas do próprio INPE e não foram misturadas com o INMET.


## Sugestão de gráficos para a apresentação

Todos são salvos em `MyDrive/figuras/`. Para uma apresentação curta, eu usaria estes 5, nesta ordem:

1. `06_01_volume_bruto_vs_normalizado.png`: abre a história (o Cerrado domina em número, o Pantanal em densidade).
2. `06_02_sazonalidade_comparada.png`: mostra que a época é a mesma.
3. `06_03_variabilidade_entre_anos.png`: mostra a diferença real (Pantanal irregular, Cerrado constante), com 2020 em destaque.
4. `06_05_prob_foco_por_clima.png`: a resposta sobre clima × fogo.
5. `06_06_cobertura_estacoes.png` (ou o mapa `06_07_mapa_biomas.png`): a limitação principal, com honestidade.

Como apoio, ficam o FRP (`06_04`) e os gráficos de cada bioma dos notebooks 04 e 05.
